In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, BooleanType, TimestampType
)

In [0]:
# 1. Configuration & Unity Catalog Paths
CATALOG = "workspace"
SCHEMA = "fraud_lakehouse"

RAW_LANDING_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_landing_volume/"
CHECKPOINT_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints_volume/bronze_transactions"
SCHEMA_LOCATION = f"{CHECKPOINT_BASE}/_schema"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_transactions"
DLQ_TABLE = f"{CATALOG}.{SCHEMA}.bronze_transactions_dlq"

In [0]:
# 2. Define Strict Ingestion Contract (Schema)
transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("account_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("txn_type", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("merchant_category", StringType(), True),
    StructField("location_country", StringType(), True),
    StructField("event_timestamp", TimestampType(), True),
    StructField("is_suspicious_flag", BooleanType(), True)
])

In [0]:


# 3. Read Stream via Databricks Auto Loader (cloudFiles)
raw_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    # 'rescue' mode captures type mismatches & unexpected columns in _rescued_data instead of failing
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(transaction_schema)
    .load(RAW_LANDING_PATH)
    # Attach lineage & audit metadata
    .withColumn("_ingest_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

In [0]:
# 4. Micro-Batch Router: Split Clean Records vs. Poison Pills (DLQ)
def route_bronze_and_dlq(micro_batch_df, batch_id):
    # Persist micro-batch in memory to avoid reading source files twice
    micro_batch_df.persist()
    
    # Define Poison Pill / Contract Violation Criteria:
    # - _rescued_data is not null (e.g., amount was string "INVALID_AMOUNT", or unknown keys appeared)
    # - Critical primary/business keys are null (transaction_id, account_id, amount)
    # - Negative transaction amounts
    is_corrupted = (
        F.col("_rescued_data").isNotNull() |
        F.col("transaction_id").isNull() |
        F.col("account_id").isNull() |
        F.col("amount").isNull() |
        (F.col("amount") <= 0)
    )
    
    valid_df = micro_batch_df.filter(~is_corrupted).drop("_rescued_data")
    
    dlq_df = (
        micro_batch_df.filter(is_corrupted)
        .withColumn(
            "_dlq_reason",
            F.when(F.col("_rescued_data").isNotNull(), "SCHEMA_OR_TYPE_MISMATCH")
             .when(F.col("transaction_id").isNull(), "MISSING_PRIMARY_KEY")
             .when(F.col("amount").isNull() | (F.col("amount") <= 0), "INVALID_TRANSACTION_AMOUNT")
             .otherwise("UNKNOWN_CONTRACT_VIOLATION")
        )
        .withColumn("_batch_id", F.lit(batch_id))
    )
    
    # Write clean records to Bronze Delta Table
    if not valid_df.isEmpty():
        (
            valid_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(BRONZE_TABLE)
        )
        
    # Quarantine corrupted records to Dead-Letter Queue Table
    if not dlq_df.isEmpty():
        (
            dlq_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(DLQ_TABLE)
        )
        
    micro_batch_df.unpersist()

In [0]:
# 5. Execute Trigger.AvailableNow (Serverless-optimized incremental batch)
query = (
    raw_stream_df.writeStream
    .foreachBatch(route_bronze_and_dlq)
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/_checkpoint")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print("Bronze ingestion and DLQ routing completed successfully.")

In [0]:
%sql
-- Verify split between valid Bronze records and quarantined DLQ records
SELECT 'BRONZE_VALID' AS layer, COUNT(*) AS record_count FROM workspace.fraud_lakehouse.bronze_transactions
UNION ALL
SELECT 'DLQ_QUARANTINED' AS layer, COUNT(*) AS record_count FROM workspace.fraud_lakehouse.bronze_transactions_dlq;

In [0]:
%sql
SELECT 
  _dlq_reason, 
  COUNT(*) AS count, 
  FIRST(_rescued_data) AS sample_rescued_payload
FROM workspace.fraud_lakehouse.bronze_transactions_dlq
GROUP BY _dlq_reason;